# 01. EDA - 스마트 창고 출고 지연 예측

## 대회 정보
- **주제**: 향후 30분간 평균 출고 지연 시간(분) 예측
- **평가지표**: MAE
- **타겟**: `avg_delay`
- **피처**: 90개 (로봇 상태, 주문량, 배터리, 혼잡도 등)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded!')

## 1. 데이터 로드

In [ ]:
DATA_PATH = '../data/'

train = pd.read_csv(DATA_PATH + 'train.csv')
test = pd.read_csv(DATA_PATH + 'test.csv')
layout = pd.read_csv(DATA_PATH + 'layout_info.csv')
submission = pd.read_csv(DATA_PATH + 'sample_submission.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Layout shape: {layout.shape}')
print(f'Submission shape: {submission.shape}')

## 2. 기본 정보 확인

In [ ]:
print('=== Train Info ===')
print(train.info())
print('\n=== Train Describe ===')
train.describe()

In [ ]:
print('=== Train Head ===')
train.head()

In [ ]:
print('=== Test Head ===')
test.head()

In [ ]:
print('=== Layout Info ===')
print(layout.info())
print('\n')
layout.head()

## 3. 결측치 분석

In [ ]:
# 결측치 비율 확인
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)

if len(missing_train) > 0:
    print(f'결측치가 있는 컬럼 수: {len(missing_train)}')
    print('\n결측치 비율 (%):')
    print((missing_train / len(train) * 100).round(2))
else:
    print('결측치 없음!')

## 4. 타겟 변수 분석 (avg_delay)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 분포
axes[0].hist(train['avg_delay'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_title('avg_delay 분포')
axes[0].set_xlabel('avg_delay (분)')
axes[0].set_ylabel('빈도')

# 박스플롯
axes[1].boxplot(train['avg_delay'])
axes[1].set_title('avg_delay 박스플롯')

# 로그 변환 분포
axes[2].hist(np.log1p(train['avg_delay']), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[2].set_title('log(avg_delay+1) 분포')
axes[2].set_xlabel('log(avg_delay+1)')

plt.tight_layout()
plt.show()

print(f'\n타겟 통계:')
print(train['avg_delay'].describe())

## 5. 피처 분석

In [ ]:
# 수치형 피처 목록
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = train.select_dtypes(include=['object']).columns.tolist()

print(f'수치형 피처 수: {len(num_cols)}')
print(f'범주형 피처 수: {len(cat_cols)}')

if cat_cols:
    print(f'\n범주형 피처: {cat_cols}')

In [ ]:
# 타겟과의 상관관계 Top 20
if 'avg_delay' in train.columns:
    corr_with_target = train[num_cols].corr()['avg_delay'].drop('avg_delay').abs().sort_values(ascending=False)
    
    plt.figure(figsize=(12, 8))
    corr_with_target.head(20).plot(kind='barh', color='steelblue')
    plt.title('avg_delay와 상관관계 Top 20 피처')
    plt.xlabel('|상관계수|')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# 상위 상관 피처들의 산점도
top_features = corr_with_target.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, feat in enumerate(top_features):
    row, col = idx // 3, idx % 3
    axes[row][col].scatter(train[feat], train['avg_delay'], alpha=0.3, s=5)
    axes[row][col].set_xlabel(feat)
    axes[row][col].set_ylabel('avg_delay')
    axes[row][col].set_title(f'{feat} vs avg_delay')

plt.tight_layout()
plt.show()

## 6. 시나리오/타임슬롯 분석

In [ ]:
# 시나리오 ID, 타임슬롯 관련 컬럼 확인
id_like_cols = [c for c in train.columns if any(kw in c.lower() for kw in ['id', 'scenario', 'time', 'slot', 'step'])]
print('ID/시나리오/타임 관련 컬럼:', id_like_cols)

for col in id_like_cols:
    print(f'\n{col}: unique={train[col].nunique()}, min={train[col].min()}, max={train[col].max()}')

## 7. Layout 보조 테이블 분석

In [ ]:
print('=== Layout 컬럼 ===')
print(layout.columns.tolist())
print(f'\nLayout shape: {layout.shape}')
print('\n=== Layout Describe ===')
layout.describe()

## 8. Train/Test 분포 비교

In [ ]:
# 주요 피처의 train/test 분포 비교
compare_cols = corr_with_target.head(6).index.tolist() if 'corr_with_target' in dir() else num_cols[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for idx, col in enumerate(compare_cols):
    row, col_idx = idx // 3, idx % 3
    axes[row][col_idx].hist(train[col], bins=50, alpha=0.5, label='train', density=True)
    axes[row][col_idx].hist(test[col], bins=50, alpha=0.5, label='test', density=True)
    axes[row][col_idx].set_title(col)
    axes[row][col_idx].legend()

plt.suptitle('Train vs Test 분포 비교', fontsize=14)
plt.tight_layout()
plt.show()

## 9. 요약 및 인사이트

### TODO: 데이터 확인 후 작성
- [ ] 타겟 분포 특성 (정규성, 이상치)
- [ ] 주요 피처 파악
- [ ] 결측치 처리 전략
- [ ] layout_info 활용 방안
- [ ] 피처 엔지니어링 아이디어